# Tarea 1 — Del perceptrón a la MLP
## Diseño inicial de una red neuronal para el reto semafórico

**Unidad de formación:** TC2035 — Diseño de Redes Neuronales y Aprendizaje Profundo  
**Reto:** Predicción inteligente de tiempos de verde  
**Modalidad:** Individual  
**Entorno:** Google Colab / Jupyter Notebook

### Propósito
Aplicar los conceptos de perceptrón, propagación hacia adelante (feedforward), perceptrón multicapa (MLP), funciones de activación, conteo de parámetros y normalización al problema de una intersección semaforizada.

En esta actividad **no se entrena un modelo**: se diseña una arquitectura y se justifican sus decisiones. Las gráficas y comprobaciones se generan ejecutando las celdas en orden.

**Contenido:** (1) Softmax, (2) diseño de una MLP, (3) activaciones, (4) Min-Max y (5) reflexión operativa.

### Preparación del entorno

Se utilizan NumPy, Pandas y Matplotlib para cálculos y visualizaciones. Scikit-learn permite aplicar Min-Max, y TensorFlow/Keras se utiliza solo para construir la arquitectura y revisar el resumen.

En Google Colab estas bibliotecas suelen estar disponibles. En un Jupyter local, si falta alguna dependencia, es necesario instalarla en el entorno correspondiente.

In [ ]:
# Bibliotecas generales y semilla para reproducibilidad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

np.set_printoptions(precision=6, suppress=True)
pd.set_option("display.precision", 4)
np.random.seed(42)

print("Entorno preparado. NumPy:", np.__version__)

---

## Parte 1. Ejercicio de Softmax

El vector de logits (puntajes todavía sin interpretar como probabilidades) es:

\[
\mathbf{z}=[2,\;4.3,\;5,\;-1].
\]

### 1. Cálculo manual

La función Softmax transforma cada puntaje en una probabilidad:

\[
\operatorname{Softmax}(z_i)=
\frac{e^{z_i}}{\sum_{j=1}^{4}e^{z_j}}.
\]

Primero calculamos las exponenciales:

\[
\begin{aligned}
e^{2}&\approx 7.389056,\\
e^{4.3}&\approx 73.699794,\\
e^{5}&\approx 148.413159,\\
e^{-1}&\approx 0.367879.
\end{aligned}
\]

Por tanto, el denominador común es:

\[
S=7.389056+73.699794+148.413159+0.367879
\approx 229.869888.
\]

Dividiendo cada término por ese mismo denominador:

\[
\boxed{\operatorname{Softmax}(\mathbf{z})\approx
[0.032145,\ 0.320615,\ 0.645640,\ 0.001600]}.
\]

Son aproximadamente **3.21 %, 32.06 %, 64.56 % y 0.16 %**.

La forma numéricamente estable resta el máximo puntaje a todos los valores:

\[
p_i=\frac{e^{z_i-\max(\mathbf{z})}}
{\sum_j e^{z_j-\max(\mathbf{z})}}.
\]

Esta resta no altera las probabilidades, ya que el factor común se cancela; reduce el riesgo de desbordamiento al trabajar con números grandes.

In [ ]:
# Comprobación de Softmax mediante NumPy (versión estable)
z = np.array([2.0, 4.3, 5.0, -1.0], dtype=float)

exp_z = np.exp(z - np.max(z))
probabilidades = exp_z / np.sum(exp_z)

tabla_softmax = pd.DataFrame({
    "Clase": ["Clase 1", "Clase 2", "Clase 3", "Clase 4"],
    "Logit": z,
    "Probabilidad": probabilidades,
    "Porcentaje (%)": 100 * probabilidades
})
display(tabla_softmax.round(6))

print(f"Suma de probabilidades: {probabilidades.sum():.12f}")
print(f"Clase predicha: {np.argmax(probabilidades) + 1}")
assert np.isclose(probabilidades.sum(), 1.0)
assert np.argmax(probabilidades) == 2

### 3. Gráfica de probabilidades

Cada barra representa la probabilidad estimada para una de las cuatro clases. Los valores son un **ejemplo matemático**; todavía no se han asociado a condiciones de tráfico concretas.

In [ ]:
# Visualización de la distribución Softmax
nombres_clases = ["Clase 1", "Clase 2", "Clase 3", "Clase 4"]
fig, ax = plt.subplots(figsize=(8, 4.5))
barras = ax.bar(
    nombres_clases, probabilidades,
    color=["#87a7d5", "#6496cd", "#2377b5", "#a5b7cb"],
    edgecolor="#334155"
)
for barra, valor in zip(barras, probabilidades):
    ax.text(barra.get_x() + barra.get_width()/2, valor + 0.015,
            f"{valor:.2%}", ha="center", va="bottom", fontsize=10)
ax.set_ylim(0, 0.76)
ax.set_ylabel("Probabilidad")
ax.set_title("Probabilidades obtenidas con Softmax")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

### 4. ¿Cuál es la clase predicha?

La **clase 3** porque su logit es el mayor (\(5\)) y, en consecuencia, también tiene la mayor probabilidad (\(\approx 64.56\%\)). Se selecciona mediante la operación \(\arg\max_i p_i\). La clase 3 no tiene nombre de tráfico específico, pues no se proporcionó un mapeo de etiquetas.

### 5. ¿Por qué las probabilidades suman uno?

Todas las probabilidades usan el mismo denominador, que es la suma de las exponenciales:

\[
\sum_{i=1}^{4}p_i
=\sum_{i=1}^{4}\frac{e^{z_i}}{\sum_{j=1}^{4}e^{z_j}}
=\frac{\sum_{i=1}^{4}e^{z_i}}{\sum_{j=1}^{4}e^{z_j}}
=1.
\]

Además, cada exponencial es positiva, por lo que todas las probabilidades se encuentran entre 0 y 1.

### Pregunta de reflexión: Softmax frente a una salida lineal

Si se quiere **clasificar cuatro condiciones de tráfico** (por ejemplo, bajo, moderado, alto y crítico), Softmax entrega una distribución de cuatro probabilidades que suman 1. Se selecciona una clase o se usan las probabilidades para apoyar una decisión.

Si se desean **predecir cuatro tiempos de verde**, se necesitan cuatro valores numéricos en segundos que no tienen por qué sumar 1. Una salida lineal con cuatro neuronas permite representar estos valores de forma independiente (aunque pueden resultar negativos y se deben imponer restricciones).

**En este reto simplificado las salidas son dos tiempos (NS y EO), no cuatro**. Por eso se diseña una MLP con **dos neuronas lineales de salida**. Softmax sobre dos o cuatro tiempos impondría una suma de 1 que no corresponde directamente a segundos.

---

## Parte 2. Arquitectura de una MLP

### De un perceptrón a una red feedforward multicapa

Un perceptrón calcula una combinación ponderada de entradas más un sesgo, seguida de una activación: \(a=f(\mathbf{w}^{\top}\mathbf{x}+b)\). Una **MLP** conecta múltiples neuronas en capas; así puede aprender relaciones no lineales entre volumen/ocupación y tiempos sugeridos.

El flujo es hacia adelante: **entradas → capa oculta 1 → capa oculta 2 → salidas**. No hay conexiones recurrentes en este diseño.

**Entradas (4 variables):**
1. Volumen vehicular norte-sur (NS).
2. Volumen vehicular este-oeste (EO).
3. Ocupación NS.
4. Ocupación EO.

**Salidas (2 variables continuas):**
1. Tiempo de verde NS, en segundos.
2. Tiempo de verde EO, en segundos.

Arquitectura solicitada:

\[
\boxed{4\longrightarrow16\longrightarrow8\longrightarrow2}
\]

Se propone ReLU en las dos capas ocultas y activación lineal en la salida; esta elección se analiza en la Parte 3.

### 1. Dibujo de la arquitectura

El siguiente diagrama se genera con Matplotlib. Las líneas representan conexiones entre todas las neuronas de capas consecutivas; las posiciones son ilustrativas, **no pesos aprendidos**.

In [ ]:
# Diagrama de la red totalmente conectada 4–16–8–2
tam_capas = [4, 16, 8, 2]
titulos = ["Entrada", "Oculta 1 (ReLU)", "Oculta 2 (ReLU)", "Salida (lineal)"]
colores = ["#6e9bc5", "#7ba7a1", "#7ba7a1", "#d5a35b"]
xs = [0, 3, 6, 9]

fig, ax = plt.subplots(figsize=(13, 7))
posiciones = []
for x, n in zip(xs, tam_capas):
    ys = np.linspace(-(n - 1)/2, (n - 1)/2, n)
    # Misma altura total de dibujo para redes de distinto tamaño.
    if n > 1:
        ys = ys / ((max(tam_capas) - 1)/2) * 2.7
    posiciones.append([(x, y) for y in ys])

# Conexiones de cada par de capas consecutivas
for anterior, siguiente in zip(posiciones[:-1], posiciones[1:]):
    for xa, ya in anterior:
        for xb, yb in siguiente:
            ax.plot([xa, xb], [ya, yb], color="#8a96a5",
                    alpha=0.20, linewidth=0.7, zorder=1)

for i, (grupo, titulo, color) in enumerate(zip(posiciones, titulos, colores)):
    xp, yp = zip(*grupo)
    ax.scatter(xp, yp, s=360, c=color, edgecolors="#334155", linewidths=1.1, zorder=3)
    ax.text(xs[i], 3.25, f"{titulo}\n{tam_capas[i]} neuronas",
            ha="center", va="center", fontsize=11, fontweight="bold")

# Variables de entrada y salida como anotaciones del diagrama
entradas = ["Volumen NS", "Volumen EO", "Ocupación NS", "Ocupación EO"]
for (x, y), etiqueta in zip(posiciones[0], entradas):
    ax.text(x - 0.32, y, etiqueta, ha="right", va="center", fontsize=9)
salidas = ["Verde NS (s)", "Verde EO (s)"]
for (x, y), etiqueta in zip(posiciones[-1], salidas):
    ax.text(x + 0.35, y, etiqueta, ha="left", va="center", fontsize=10)

ax.set_xlim(-2.3, 11)
ax.set_ylim(-3.35, 3.65)
ax.axis("off")
ax.set_title("Arquitectura MLP 4–16–8–2", fontsize=15, pad=18)
plt.tight_layout()
plt.show()

### 2–4. Conteo manual de pesos, sesgos y parámetros

Para una capa densa completamente conectada con \(n_{\mathrm{in}}\) entradas y \(n_{\mathrm{out}}\) neuronas:

\[
N_{\mathrm{pesos}}=n_{\mathrm{in}}n_{\mathrm{out}},\qquad
N_{\mathrm{sesgos}}=n_{\mathrm{out}},\qquad
N_{\mathrm{parámetros}}=(n_{\mathrm{in}}+1)n_{\mathrm{out}}.
\]

| Capa densa | Pesos | Sesgos | Parámetros |
|:---|---:|---:|---:|
| Entrada (4) → Oculta 1 (16) | \(4\times16=64\) | 16 | 80 |
| Oculta 1 (16) → Oculta 2 (8) | \(16\times8=128\) | 8 | 136 |
| Oculta 2 (8) → Salida (2) | \(8\times2=16\) | 2 | 18 |
| **Total** | **208** | **26** | **234** |

Por tanto:

\[
\boxed{N_{\text{total}}=64+128+16+16+8+2=234}.
\]

Las **cuatro entradas no añaden parámetros entrenables** por sí mismas. Cada neurona densa sí tiene un sesgo adicional.

In [ ]:
# Verificación del conteo matemático sin depender de Keras
dimensiones = [4, 16, 8, 2]
pesos_por_capa = [a*b for a, b in zip(dimensiones[:-1], dimensiones[1:])]
sesgos_por_capa = dimensiones[1:]
total_pesos = sum(pesos_por_capa)
total_sesgos = sum(sesgos_por_capa)
total_parametros = total_pesos + total_sesgos

tabla_parametros = pd.DataFrame({
    "Capa": ["Densa 1", "Densa 2", "Densa 3"],
    "Pesos": pesos_por_capa,
    "Sesgos": sesgos_por_capa,
    "Parámetros": np.array(pesos_por_capa) + np.array(sesgos_por_capa)
})
display(tabla_parametros)
print(f"Pesos: {total_pesos} | Sesgos: {total_sesgos} | Total: {total_parametros}")
assert (total_pesos, total_sesgos, total_parametros) == (208, 26, 234)

### 5. Construcción en TensorFlow/Keras

Se define una entrada de cuatro características y tres capas densas de 16, 8 y 2 neuronas. En esta actividad **no se compila ni entrena** la red, porque no se proporcionó un conjunto de pares entrada–tiempo de verde objetivo.

El resumen de Keras debe mostrar **80 + 136 + 18 = 234 parámetros entrenables**. Los nombres de las capas pueden variar según la versión de TensorFlow.

In [ ]:
# Construcción de la MLP y validación con model.summary()
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)

model = keras.Sequential([
    keras.Input(shape=(4,), name="entradas_trafico"),
    layers.Dense(16, activation="relu", name="oculta_1"),
    layers.Dense(8, activation="relu", name="oculta_2"),
    layers.Dense(2, activation="linear", name="tiempos_verde")
], name="MLP_reto_semaforico")

model.summary()

# Verificación automática del número de parámetros
assert model.count_params() == 234, "El conteo debe ser de 234 parámetros."
for capa in model.layers:
    print(f"{capa.name}: {capa.count_params()} parámetros")
print("Total comprobado con Keras:", model.count_params())

---

## Parte 3. Selección de funciones de activación

### 1. ¿Qué activación utilizaría en las capas ocultas?

Utilizaría **ReLU**, definida por \(f(x)=\max(0,x)\). Introduce no linealidad y permite modelar relaciones que un único perceptrón lineal no podría representar, por ejemplo, que el efecto del volumen dependa también de la ocupación. Es sencilla de calcular y habitualmente facilita la optimización. Se debe vigilar el problema de neuronas que dejan de activarse; si aparece, puede probarse Leaky ReLU.

### 2. ¿Qué activación utilizaría en la salida para predecir segundos de verde?

Utilizaría una **salida lineal** (\(f(x)=x\)) en cada una de las dos neuronas para estimar dos variables continuas: segundos de verde NS y EO. Al no limitar la salida a [0, 1], ofrece flexibilidad para la regresión. Sin embargo, puede producir tiempos negativos o demasiado grandes; por ello **no debe enviarse directamente a un controlador real**.

Si se conocen límites operativos fijos, pueden incorporarse restricciones mediante una transformación de salida adecuada o un módulo de asignación factible, y después evaluar su efecto sobre el rendimiento del sistema.

### 3. ¿Por qué Softmax no sería automáticamente la mejor opción?

Softmax genera probabilidades no negativas cuya suma es 1. Los segundos de verde **no son clases mutuamente excluyentes ni probabilidades**, y sus valores no necesitan sumar 1. Una distribución Softmax podría servir para elegir entre planes discretos predefinidos, pero no entrega por sí sola duraciones continuas en segundos.

### 4. ¿Qué restricciones operativas deberían revisarse?

- **Duraciones mínimas y máximas** de verde para cada movimiento y límites de tiempo de ciclo.
- **Seguridad entre fases**: intervalos de amarillo, todo-rojo y evitar verdes incompatibles.
- **Cruces peatonales**: tiempos mínimos de cruce y accesibilidad.
- **Secuencias y coordinación**: movimientos permitidos, prioridades, giros, coordinación con intersecciones cercanas y normas del controlador.
- **Demanda y equidad**: impedir esperas excesivas y atender movimientos con menor volumen, transporte público o emergencias según la política aplicable.
- **Robustez y contingencias**: fallas de sensores, valores fuera de rango y un plan seguro de respaldo.

**Importante:** esta MLP solo propone tiempos; una capa de validación o de optimización con restricciones debe decidir si esos tiempos son admisibles antes de aplicarlos.

---

## Parte 4. Normalización de variables

Los volúmenes se expresan en conteos de vehículos, mientras que las ocupaciones son fracciones entre 0 y 1. Como están en escalas distintas, sus magnitudes podrían afectar de manera desigual el proceso de ajuste de los pesos.

Para cada variable \(x\) se aplica la transformación Min-Max:

\[
x'=\frac{x-x_{\min}}{x_{\max}-x_{\min}}.
\]

En este ejemplo se usan los cuatro escenarios didácticos para calcular mínimos y máximos. **En un proyecto real el escalador debe ajustarse solo con el conjunto de entrenamiento** y después aplicarse, sin volver a ajustarlo, a validación, prueba y producción.

In [ ]:
# 1. Registro de los cuatro escenarios originales
datos = pd.DataFrame({
    "Escenario": ["A", "B", "C", "D"],
    "Volumen NS": [300, 600, 900, 450],
    "Volumen EO": [700, 400, 300, 650],
    "Ocupación NS": [0.25, 0.55, 0.80, 0.40],
    "Ocupación EO": [0.65, 0.35, 0.20, 0.60]
}).set_index("Escenario")

print("Datos originales:")
display(datos)

### 2. Valores mínimos y máximos

| Variable | Mínimo | Máximo |
|:---|---:|---:|
| Volumen NS | 300 | 900 |
| Volumen EO | 300 | 700 |
| Ocupación NS | 0.25 | 0.80 |
| Ocupación EO | 0.20 | 0.65 |

Ejemplo manual para el escenario B, volumen NS:

\[
x'_{B,\mathrm{VolNS}}=\frac{600-300}{900-300}=\frac{300}{600}=0.5.
\]

Y para ocupación EO del escenario B:

\[
x'_{B,\mathrm{OcuEO}}=\frac{0.35-0.20}{0.65-0.20}
=\frac{0.15}{0.45}\approx0.3333.
\]

Con el escalado, el escenario A se convierte en [0, 1, 0, 1], y el C en [1, 0, 1, 0] según el orden de columnas.

In [ ]:
# 2–3. Ajuste de Min-Max y comparación de datos originales y transformados
from sklearn.preprocessing import MinMaxScaler

escalador = MinMaxScaler(feature_range=(0, 1))
matriz_normalizada = escalador.fit_transform(datos)  # SOLO para ejercicio didáctico

datos_normalizados = pd.DataFrame(
    matriz_normalizada, columns=datos.columns, index=datos.index
)

print("Valores originales:")
display(datos)
print("Valores normalizados (Min-Max):")
display(datos_normalizados.round(4))

# Una tabla conjunta facilita comparar cada valor original con su escala normalizada
comparacion = pd.concat(
    [datos.add_suffix(" (original)"), datos_normalizados.add_suffix(" (Min-Max)")],
    axis=1
)
print("Comparación por escenario:")
display(comparacion.round(4))

assert np.all(matriz_normalizada >= -1e-12)
assert np.all(matriz_normalizada <= 1 + 1e-12)
assert np.allclose(matriz_normalizada.min(axis=0), 0)
assert np.allclose(matriz_normalizada.max(axis=0), 1)

### 3. Gráfica de los valores transformados

El mapa de calor representa la magnitud normalizada de cada variable para los cuatro escenarios. Un 0 corresponde al mínimo observado y un 1 al máximo observado **dentro de esta pequeña tabla**.

In [ ]:
# Mapa de calor sin bibliotecas adicionales
fig, ax = plt.subplots(figsize=(9, 4.3))
im = ax.imshow(datos_normalizados.to_numpy(), cmap="Blues", vmin=0, vmax=1, aspect="auto")

ax.set_xticks(np.arange(len(datos.columns)), labels=datos.columns)
ax.set_yticks(np.arange(len(datos.index)), labels=datos.index)
ax.set_xlabel("Variable")
ax.set_ylabel("Escenario")
ax.set_title("Normalización Min-Max por escenario")
for i in range(len(datos.index)):
    for j in range(len(datos.columns)):
        valor = datos_normalizados.iloc[i, j]
        ax.text(j, i, f"{valor:.2f}", ha="center", va="center",
                color="white" if valor > 0.55 else "#172554", fontsize=10)

fig.colorbar(im, ax=ax, label="Valor normalizado (0 a 1)")
plt.tight_layout()
plt.show()

### 4. ¿Por qué el escalado ayuda a una MLP?

En una capa densa, la operación básica es \(z=\mathbf{w}^{\top}\mathbf{x}+b\). Si una variable toma valores de cientos y otra valores decimales, la primera puede dominar inicialmente las sumas y modificar la escala de los gradientes. Al llevarlas a rangos comparables, el ajuste de pesos suele resultar más estable y puede converger más rápido.

**Precauciones:**
- El ajuste del escalador (mínimos y máximos) usa exclusivamente las observaciones de **entrenamiento** en el proyecto real, para evitar fuga de información (data leakage).
- Se reutiliza ese mismo escalador para transformar datos de validación, prueba y operación.
- Un dato nuevo fuera del rango observado en entrenamiento puede transformarse a un valor menor que 0 o mayor que 1; eso no es necesariamente un error y debe manejarse.
- Cuatro escenarios son insuficientes para entrenar y validar una MLP real; solo ilustran la transformación.

Ejemplo del procedimiento correcto con una división de datos real:

1. Dividir temporalmente o por escenarios en entrenamiento, validación y prueba, según el problema.
2. Aplicar ajuste del escalador **solo a X_train**.
3. Aplicar transformación a X_train, X_val y X_test con **el mismo escalador**.
4. Almacenar el escalador junto con el modelo para la inferencia.

---

## Parte 5. Reflexión sobre el reto semafórico

### ¿Predecir bien tiempos de referencia equivale a reducir demoras?

**No.** Son dos objetivos de evaluación distintos.

Una red que predice correctamente los tiempos de verde de referencia reproduce las decisiones presentes en los datos históricos, por ejemplo, las recomendaciones de un operador o de una política semafórica existente. Esto se mide con métricas de regresión, como **MAE** y **RMSE** entre los tiempos predichos y los tiempos objetivo. Sin embargo, esas referencias no necesariamente son óptimas: imitar un sistema que genera filas largas puede reproducir sus ineficiencias.

Para demostrar una **reducción de demoras**, debe evaluarse qué ocurre cuando se aplican las propuestas de verde en la dinámica del tráfico. El cambio de tiempos altera colas, llegadas, tiempos de espera y el servicio disponible para movimientos en conflicto. Una buena predicción estadística no demuestra automáticamente una mejor operación.

### Datos y restricciones adicionales necesarios

**Datos operativos:** flujos por movimiento y carril, velocidades, ocupaciones, colas, llegadas en el tiempo, peatones, giros, tiempos reales de fases, longitud de ciclo, saturación, horarios, incidentes y coordinación con cruces vecinos.

**Restricciones:** tiempos mínimos y máximos por fase, seguridad de cruces, amarillo y todo-rojo, pasos peatonales, compatibilidad de movimientos, máximos de espera, límites del controlador y reglas locales aplicables.

**Experimentos para validar:** comparar la política propuesta con un semáforo de tiempo fijo y, cuando exista, un esquema actuado o adaptativo. Usar escenarios de tráfico de baja y alta demanda, horas pico, incidentes y cambios de distribución. Hacer la comparación en simulación microscópica calibrada (por ejemplo, SUMO) antes de considerar una prueba de campo autorizada con mecanismos de seguridad.

**Métricas operativas:** demora promedio y por movimiento, percentiles de espera, longitud máxima de cola, número de detenciones, vehículos atendidos, tiempos peatonales y posibles movimientos perjudicados. Conviene reportar también incertidumbre entre simulaciones, por ejemplo, con distintas semillas y días de demanda.

**Diseño de evaluación:** separar escenarios de prueba de los usados para desarrollar el método, mantener la misma demanda para cada política comparada y analizar no solo la media, sino también si alguna dirección sufre demoras desproporcionadas.

Solo una evaluación que cumpla estas condiciones aportaría evidencia de si el modelo **mejora el funcionamiento real**; un modelo predictivo aislado no es suficiente para esa afirmación.

---

## Conclusión personal

Esta actividad me ayudó a entender cómo se pasa de un perceptrón a una MLP y por qué el diseño depende del tipo de problema. Con Softmax comprobé que una salida de clasificación representa probabilidades, mientras que, para predecir tiempos de verde, necesito valores continuos en segundos. También aprendí a calcular los pesos y sesgos de una red antes de construirla en Keras y a verificar que la arquitectura propuesta tiene **234 parámetros**.

Lo que considero más importante para el reto semafórico es que **un modelo preciso no necesariamente es una buena solución de tráfico**. Aunque consiga reproducir tiempos de referencia, todavía tendría que verificar que sus propuestas son seguras y que, al aplicarlas, realmente disminuyen las demoras sin perjudicar otras direcciones o a los peatones. Por eso, antes de pensar en su implementación, validaría las restricciones y compararía sus resultados con otras estrategias mediante simulaciones y pruebas controladas.

In [ ]:
# Comprobaciones finales de las partes que no requieren entrenamiento
assert len(z) == 4
assert np.isclose(probabilidades.sum(), 1.0)
assert np.argmax(probabilidades) == 2
assert total_parametros == 234
assert model.count_params() == 234
assert datos.shape == (4, 4)
assert datos_normalizados.shape == (4, 4)
assert not datos_normalizados.isna().any().any()

print("Comprobaciones satisfactorias:")
print("- Softmax: 4 probabilidades que suman 1.")
print("- Clase predicha: 3.")
print("- MLP: 234 parámetros según cálculo y Keras.")
print("- Datos: 4 escenarios y 4 variables correctamente normalizados.")
print("Notebook completo. No se ha entrenado una red.")